# CircuitLM Training — Personal Dataset

Trains a hybrid CircuitLM (PDA circuit + neural corrector) on your personal conversations.
Runs on Kaggle GPU. Loads data from HuggingFace private dataset.

**Runtime:** GPU (P100/T4). ~15-30 min for full training.

## 1. Install Dependencies

In [ ]:
%pip install orjson sentencepiece torch --quiet
import subprocess, sys
print('Installed')

## 2. Load Personal Dataset from HuggingFace

In [ ]:
from datasets import load_dataset
import os

# Load from your private HF dataset
ds = load_dataset("toxzak/circuit-lm-personal", split="train")
print(f"Loaded: {len(ds)} rows")
print(ds[0])

## 3. Build Training Data

Use the combined `research_evolver_data.txt` which is deduplicated.

In [ ]:
# Combine all text into one file
all_text = ""
for row in ds:
    text = row.get('text', row.get('content', ''))
    if text:
        all_text += text + "\n"

# Use the combined dataset (research_evolver_data.txt has 21K lines)
lines = all_text.strip().split('\n')
print(f"Total lines: {len(lines)}")

# Subsample for speed (use all for final training)
MAX_LINES = 15000  # Use 15K lines for GPU training speed
lines = lines[:MAX_LINES]
train_text = "\n".join(lines)
print(f"Using {len(lines)} lines for training")

## 4. Add CircuitLM src to Path

In [ ]:
# Clone circuit_lm repo and add to path
!git clone https://github.com/toxzak-svg/circuit_lm.git /tmp/circuit_lm
import sys
sys.path.insert(0, '/tmp/circuit_lm/src')
print('circuit_lm cloned and in path')

## 5. Tokenizer + Data Prep

In [ ]:
import time
from circuit_lm.tokenizer import Tokenizer
from circuit_lm.data import load_sequences

VOCAB_SIZE = 4096

print('Building BPE tokenizer...')
t0 = time.time()
tokenizer = Tokenizer.from_text(
    train_text,
    vocab_size=VOCAB_SIZE,
    mode='bpe',
    bpe_merges=VOCAB_SIZE,
)
print(f'  vocab={tokenizer.vocab_size}, took {time.time()-t0:.1f}s')

print('Tokenizing sequences...')
t0 = time.time()
sequences = load_sequences(train_text, tokenizer)
print(f'  {len(sequences)} sequences, {sum(len(s) for s in sequences)} tokens, took {time.time()-t0:.1f}s')

## 6. Train PDA Circuit (CP-SAT)

In [ ]:
from circuit_lm.train_joint_pda_cpsat import train_joint_pda as train_pda
import time

STATE_BITS = 6     # 64 states
STACK_DEPTH = 4
STACK_STEPS = 15
TRANS_STEPS = 22
EMIT_STEPS = 23   # total = 60

print('Training PDA circuit...')
t0 = time.time()
circuit = train_pda(
    sequences=sequences,
    vocab_size=tokenizer.vocab_size,
    state_bits=STATE_BITS,
    stack_depth=STACK_DEPTH,
    stack_steps=STACK_STEPS,
    transition_steps=TRANS_STEPS,
    emission_steps=EMIT_STEPS,
)
circuit_time = time.time() - t0
print(f'  Circuit trained in {circuit_time:.1f}s')

## 7. Save Circuit + Tokenizer

Circuit is saved to /tmp so you can download it.

In [ ]:
from circuit_lm.io import save_model

CIRCUIT_PATH = '/tmp/circuit_4k.json'
save_model(circuit, tokenizer, CIRCUIT_PATH)
import os
print(f'Circuit saved: {os.path.getsize(CIRCUIT_PATH)/1024:.0f} KB')

## 8. Train Hybrid Corrector

Neural corrector learns to fix circuit mistakes. Uses GPU if available.

In [ ]:
import time
import torch
from hybrid import train_hybrid

print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

CORRECTOR_PATH = '/tmp/corrector_4k.pt'
DATA_PATH = '/tmp/train_text.txt'

# Save text data for the hybrid trainer
with open(DATA_PATH, 'w') as f:
    f.write(train_text)

print('Training hybrid corrector...')
t0 = time.time()
corrector = train_hybrid(
    circuit_path=CIRCUIT_PATH,
    data_path=DATA_PATH,
    output_path=CORRECTOR_PATH,
    num_epochs=5,
    batch_size=128,
    circuit_weight=0.5,
    max_examples=80000,
)
corrector_time = time.time() - t0
print(f'Corrector trained in {corrector_time:.1f}s')

## 9. Download Results

Download circuit + corrector + tokenizer to use in Starfire.

In [ ]:
from IPython.display import FileLink
print('Circuit:')
FileLink(CIRCUIT_PATH)
print('Corrector:')
FileLink(CORRECTOR_PATH)
print()
print(f'Total training time: {circuit_time + corrector_time:.1f}s')